In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from pipeline.runner import run_full_pipeline
from readers.loaders import load_dataset, preprocess_stock_report

df = preprocess_stock_report(r'C:\Users\a.vorona\Desktop\Forecast\data\raw\Остатки-обороты.xlsx')

In [9]:
df = run_full_pipeline(r'C:\Users\a.vorona\Desktop\Forecast\data\raw\Запчасти списанные в ремонт.xlsx', r'C:\Users\a.vorona\Desktop\Forecast\data\raw\Остатки-обороты.xlsx')
df1 = run_full_pipeline(r'C:\Users\a.vorona\Desktop\Forecast\data\raw\Запчасти списанные в ремонт.xlsx', r'C:\Users\a.vorona\Desktop\Forecast\data\raw\Остатки и обороты.xlsx')
df2 = run_full_pipeline(r'C:\Users\a.vorona\Desktop\Forecast\Запчасти списанные в ремонт.xlsx', r'C:\Users\a.vorona\Desktop\Forecast\Остатки и обороты.xlsx')

In [ ]:
import sys
from collections import defaultdict
from pathlib import Path

import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import PatternFill

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from pipeline.runner import (
    _build_article_lookup,
    _drop_rows_without_identifiers,
    _enrich_analogs,
    _lookup_group,
    _sync_group_membership,
)
from preprocessing.cleaning import (
    fill_missing_article,
    normalize_nomenclatures_repair_parts,
    normalize_nomenclatures_stock_report,
)
from preprocessing.grouping import (
    build_analog_graph,
    consolidate_extended_article_numbers,
    find_all_analogs,
    normalize_analog_lists,
)
from preprocessing.normalization import article_forms, normalize
from readers.loaders import preprocess_repair_parts, preprocess_stock_report


def _export_colored_analogs_table(df: pd.DataFrame, output_path: Path) -> None:
    df.to_excel(output_path, index=False, sheet_name="analogs")

    wb = load_workbook(output_path)
    ws = wb["analogs"]

    palette = [
        "FFF2CC", "D9EAD3", "D0E0E3", "F4CCCC", "D9D2E9",
        "FCE5CD", "CFE2F3", "EAD1DC", "D9E2F3", "E2F0D9",
    ]

    current_group = None
    color_idx = -1

    for row_idx in range(2, ws.max_row + 1):
        group = ws.cell(row=row_idx, column=1).value
        if group != current_group:
            current_group = group
            color_idx += 1

        fill = PatternFill(
            fill_type="solid",
            fgColor=palette[color_idx % len(palette)],
        )
        for col_idx in range(1, ws.max_column + 1):
            ws.cell(row=row_idx, column=col_idx).fill = fill

    ws.freeze_panes = "A2"
    wb.save(output_path)


def build_analogs_table_strict(
    repair_path: str | Path,
    stock_path: str | Path,
    output_path: str | Path | None = None,
) -> pd.DataFrame:
    repair_path = str(repair_path)
    stock_path = str(stock_path)

    # 1. Repair: полностью по вашему пайплайну
    df1 = preprocess_repair_parts(repair_path)
    df1 = normalize_nomenclatures_repair_parts(df1)

    for col in [
        "Номенклатура.Артикул",
        "Номенклатура.Оригинальный номер",
        "Номенклатура.Оригинальный номер расширенный",
    ]:
        df1[col] = df1[col].apply(normalize)

    df1 = fill_missing_article(
        df1,
        "Номенклатура.Артикул",
        "Номенклатура.Оригинальный номер",
    )
    df1 = _drop_rows_without_identifiers(
        df1,
        "Номенклатура.Артикул",
        "Номенклатура.Оригинальный номер",
    )

    df1["Номенклатура.Оригинальный номер расширенный"] = df1.apply(
        consolidate_extended_article_numbers,
        axis=1,
    )
    df1["Аналоги"] = (
        df1["Номенклатура.Оригинальный номер расширенный"]
        .fillna("")
        .str.upper()
        .str.split()
    )

    graph = build_analog_graph(df1)
    df1["all_analogs"] = df1["Номенклатура.Артикул"].apply(
        lambda x: find_all_analogs(x, graph)
    )
    df1 = df1.drop(columns="Аналоги")

    group_mapping = {
        analog_tuple: idx
        for idx, analog_tuple in enumerate(df1["all_analogs"].unique(), start=1)
    }
    df1["Номер группы"] = df1["all_analogs"].apply(group_mapping.get).astype("Int64")

    # 2. Stock: полностью по вашему пайплайну
    df2 = preprocess_stock_report(stock_path)
    df2 = normalize_nomenclatures_stock_report(df2)

    df2["Артикул"] = df2["Артикул"].apply(normalize)
    df2["Оригинальный номер"] = df2["Оригинальный номер"].apply(normalize)
    df2 = fill_missing_article(df2, "Артикул", "Оригинальный номер")
    df2 = _drop_rows_without_identifiers(df2, "Артикул", "Оригинальный номер")

    article_to_group, article_to_analogs = _build_article_lookup(df1)

    groups, analogs_list = zip(
        *df2.apply(
            lambda r: _lookup_group(
                r["Артикул"],
                r["Оригинальный номер"],
                article_to_group,
                article_to_analogs,
            ),
            axis=1,
        )
    )

    df2["Номер группы"] = pd.Series(groups, index=df2.index, dtype="Int64")
    df2["Список аналогов"] = analogs_list

    df2["Список аналогов"] = df2.apply(
        lambda r: _enrich_analogs(r, article_to_group),
        axis=1,
    )
    df2 = normalize_analog_lists(df2)

    df1, df2 = _sync_group_membership(df1, df2)
    article_to_group, article_to_analogs = _build_article_lookup(df1)

    unmatched_idx = df2[df2["Номер группы"].isna()].index
    relinked_idx = []

    for idx in unmatched_idx:
        grp, analogs = _lookup_group(
            normalize(df2.at[idx, "Артикул"]),
            normalize(df2.at[idx, "Оригинальный номер"]),
            article_to_group,
            article_to_analogs,
        )
        if grp is None:
            continue

        df2.at[idx, "Номер группы"] = grp
        df2.at[idx, "Список аналогов"] = analogs
        relinked_idx.append(idx)

    if relinked_idx:
        df2.loc[relinked_idx, "Список аналогов"] = (
            df2.loc[relinked_idx]
            .apply(lambda r: _enrich_analogs(r, article_to_group), axis=1)
        )
        df2 = normalize_analog_lists(df2)
        df1, df2 = _sync_group_membership(df1, df2)
        article_to_group, article_to_analogs = _build_article_lookup(df1)

    graph_new = defaultdict(set)
    unmatched_idx = df2[df2["Номер группы"].isna()].index

    for idx in unmatched_idx:
        art = normalize(df2.at[idx, "Артикул"])
        orig = normalize(df2.at[idx, "Оригинальный номер"])

        if art is None:
            if orig is not None:
                art = orig
            else:
                continue

        all_forms = article_forms(art) + (article_forms(orig) if orig else [])
        if any(f in article_to_group for f in all_forms):
            continue

        for af in article_forms(art):
            for bf in article_forms(art):
                if af != bf:
                    graph_new[af].add(bf)
                    graph_new[bf].add(af)

        if orig is not None:
            for af in article_forms(art):
                for bf in article_forms(orig):
                    if af != bf:
                        graph_new[af].add(bf)
                        graph_new[bf].add(af)

    for idx in unmatched_idx:
        art = normalize(df2.at[idx, "Артикул"])
        orig = normalize(df2.at[idx, "Оригинальный номер"])
        if art is None:
            art = orig

        all_forms = article_forms(art) if art else []
        if any(f in article_to_group for f in all_forms):
            continue

        df2.at[idx, "Список аналогов"] = (
            find_all_analogs(art, graph_new) if art is not None else tuple()
        )

    unique_new = (
        df2.loc[df2["Номер группы"].isna(), "Список аналогов"]
        .drop_duplicates()
    )
    new_group_start = int(df1["Номер группы"].max()) + 1
    new_group_map = {
        grp: new_group_start + i
        for i, grp in enumerate(unique_new)
    }

    mask_new = df2["Номер группы"].isna()
    df2.loc[mask_new, "Номер группы"] = (
        df2.loc[mask_new, "Список аналогов"].apply(lambda x: new_group_map.get(x))
    )
    df2["Номер группы"] = df2["Номер группы"].astype("Int64")

    # 3. Плоская таблица аналогов
    repair_rows = (
        df1.loc[
            df1["Номер группы"].notna(),
            [
                "Номер группы",
                "Номенклатура.Артикул",
                "Номенклатура.Оригинальный номер",
                "Номенклатура",
                "all_analogs",
            ],
        ]
        .rename(
            columns={
                "Номенклатура.Артикул": "Артикул",
                "Номенклатура.Оригинальный номер": "Оригинальный номер",
                "all_analogs": "Список аналогов",
            }
        )
    )

    stock_rows = df2.loc[
        df2["Номер группы"].notna(),
        [
            "Номер группы",
            "Артикул",
            "Оригинальный номер",
            "Номенклатура",
            "Список аналогов",
        ],
    ]

    analogs_df = pd.concat([repair_rows, stock_rows], ignore_index=True)

    analogs_df["Список аналогов"] = analogs_df["Список аналогов"].apply(
        lambda x: ", ".join(x) if isinstance(x, tuple) else x
    )

    analogs_df = (
        analogs_df.drop_duplicates()
        .sort_values(
            by=["Номер группы", "Артикул", "Оригинальный номер", "Номенклатура"],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    # Только группы, которые встречаются больше 1 раза
    group_sizes = analogs_df.groupby("Номер группы")["Номер группы"].transform("size")
    analogs_df = analogs_df.loc[group_sizes > 1].reset_index(drop=True)

    if output_path is not None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        _export_colored_analogs_table(analogs_df, output_path)

    return analogs_df


repair_path = project_root / "data" / "raw" / "Запчасти списанные в ремонт.xlsx"
stock_path = project_root / "data" / "raw" / "Остатки и обороты.xlsx"
output_path = project_root / "data" / "processed" / "Аналоги.xlsx"

analogs_df = build_analogs_table_strict(repair_path, stock_path, output_path)
analogs_df.head(20)


,Номер группы,Артикул,Оригинальный номер,Номенклатура,Список аналогов
0,3,Я0378316,None,Разъем ВПУ в сборе с проводом Zoomlion,"12.02.000010, Z12.02.000010, Я0378316"
1,3,Я0378316,12.02.000010,Разъем ВПУ в сборе с проводом Zoomlion,"12.02.000010, Z12.02.000010, Я0378316"
2,9,1020521389,1020521389,Замок зажигания с ключом IHP-30 (1020520943) 1...,"1020521389, 26.01.000023, 26.01.000024, 413000..."
3,9,4130000723,4130000723,Замок зажигания 4130000723,"1020521389, 26.01.000023, 26.01.000024, 413000..."
4,9,4130000723001,4130000723001,Замок зажигания 4130000723001 LGMG,"1020521389, 26.01.000023, 26.01.000024, 413000..."


In [2]:
df = pd.read_excel(r'C:\Users\a.vorona\Desktop\Forecast\data\processed\Аналоги.xlsx')
df2 = pd.read_excel(r'C:\Users\a.vorona\Desktop\Forecast\data\processed\21.xlsx')

In [ ]:
import pandas as pd
from pathlib import Path

old_path = Path(r'C:\Users\a.vorona\Desktop\Forecast\data\processed\21.xlsx')
new_path = Path(r'C:\Users\a.vorona\Desktop\Forecast\data\processed\.xlsx')
report_path = Path(r"C:\Users\a.vorona\Desktop\Сравнение_аналогов.xlsx")

row_id_cols = ["Артикул", "Оригинальный номер", "Номенклатура"]
value_cols = ["Номер группы", "Список аналогов"]


def normalize_analog_list(value: str) -> str:
    parts = [p.strip() for p in str(value).split(",") if p and str(p).strip()]
    return ", ".join(sorted(set(parts)))


def load_analogs(path: Path) -> pd.DataFrame:
    df = pd.read_excel(path, sheet_name="analogs", dtype=str)
    df.columns = [str(col).strip() for col in df.columns]
    df = df.fillna("")

    required_cols = ["Номер группы", "Артикул", "Оригинальный номер", "Номенклатура", "Список аналогов"]
    for col in required_cols:
        if col not in df.columns:
            df[col] = ""

    for col in ["Артикул", "Оригинальный номер", "Номенклатура"]:
        df[col] = df[col].astype(str).str.strip()

    df["Список аналогов"] = df["Список аналогов"].apply(normalize_analog_list)
    df["Номер группы"] = pd.to_numeric(df["Номер группы"], errors="coerce").astype("Int64")

    df = df[required_cols].drop_duplicates().reset_index(drop=True)
    return df


old_df = load_analogs(old_path)
new_df = load_analogs(new_path)

comparison = old_df.merge(
    new_df,
    on=row_id_cols,
    how="outer",
    suffixes=("_old", "_new"),
    indicator=True,
)

added_rows = (
    comparison.loc[comparison["_merge"] == "right_only", row_id_cols + [f"{col}_new" for col in value_cols]]
    .rename(columns={f"{col}_new": col for col in value_cols})
    .sort_values(row_id_cols)
    .reset_index(drop=True)
)

removed_rows = (
    comparison.loc[comparison["_merge"] == "left_only", row_id_cols + [f"{col}_old" for col in value_cols]]
    .rename(columns={f"{col}_old": col for col in value_cols})
    .sort_values(row_id_cols)
    .reset_index(drop=True)
)

changed_rows = (
    comparison.loc[
        (comparison["_merge"] == "both")
        & (
            (comparison["Номер группы_old"] != comparison["Номер группы_new"])
            | (comparison["Список аналогов_old"] != comparison["Список аналогов_new"])
        ),
        row_id_cols
        + ["Номер группы_old", "Номер группы_new", "Список аналогов_old", "Список аналогов_new"],
    ]
    .sort_values(row_id_cols)
    .reset_index(drop=True)
)

summary = pd.DataFrame(
    {
        "Метрика": [
            "Строк в старом файле",
            "Строк в новом файле",
            "Добавлено строк",
            "Удалено строк",
            "Изменено строк",
            "Уникальных групп в старом",
            "Уникальных групп в новом",
        ],
        "Значение": [
            len(old_df),
            len(new_df),
            len(added_rows),
            len(removed_rows),
            len(changed_rows),
            old_df["Номер группы"].nunique(),
            new_df["Номер группы"].nunique(),
        ],
    }
)

with pd.ExcelWriter(report_path, engine="openpyxl") as writer:
    summary.to_excel(writer, sheet_name="summary", index=False)
    added_rows.to_excel(writer, sheet_name="added", index=False)
    removed_rows.to_excel(writer, sheet_name="removed", index=False)
    changed_rows.to_excel(writer, sheet_name="changed", index=False)

print(summary)
print()
print(f"Отчет сохранен в: {report_path}")

display(added_rows.head(20))
display(removed_rows.head(20))
display(changed_rows.head(20))


                     Метрика  Значение
0       Строк в старом файле       915
1        Строк в новом файле       754
2            Добавлено строк       743
3              Удалено строк       904
4             Изменено строк         6
5  Уникальных групп в старом       328
6   Уникальных групп в новом       245

Отчет сохранен в: C:\Users\a.vorona\Desktop\Сравнение_аналогов.xlsx


,Артикул,Оригинальный номер,Номенклатура,Номер группы,Список аналогов
0,00-00007899,1001740614,Фильтр масляный 1001740614,116,"00-00007899, 01174418, 1001740614, 1009806733,..."
1,00-00007900,1000002416,Фильтр топливный тонкой очистки 1000002416,243,"00-00007900, 1000002416, 1009806734, 6153, НФ-..."
2,00-00350134,185246280,Датчик давления масла,230,"00-00350134, 000005378, 185246280, 5378, P-000..."
3,00000217,1003692596B,Стартер WP3 WEICHAI,185,"00000217, 1002415445, 1002415445С, 1003692596,..."
4,000004697,000004697,Датчик угла колес 000004697,223,"(LH)0110A10BN/HA(HC), 000004697, 00004697, 000..."
5,000004697,000004697,Датчик угла колеса 000004697,223,"(LH)0110A10BN/HA(HC), 000004697, 00004697, 000..."
6,00000516A,00000516A,Фильтр гидравлический 00000516A,315,"00000516, 00000516A, 516, 516A, DH9426, R0060A..."
7,00000519,00000519,Фильтр гидравлический 00000519,205,"00000519, 00000519A, 519, 519A"
8,00000519A,00000519A,Фильтр гидравлический 00000519A,205,"00000519, 00000519A, 519, 519A"
9,00000533A,00000533A,Гидромотор 00000533A,115,"00000533, 00000533A, 00002009, 00002009A, 1010..."


,Артикул,Оригинальный номер,Номенклатура,Номер группы,Список аналогов
0,00-00007899,,Фильтр масляный 1001740614,116,"00-00007899, 01174418, 1001740614, 1009806733,..."
1,00-00007900,,Фильтр топливный тонкой очистки 1000002416,243,"00-00007900, 1000002416, 1009806734, 6153"
2,00-00350134,,Датчик давления масла,230,"00-00350134, 000005378, 185246280, 5378, P-000..."
3,00000217,,Стартер WP3 WEICHAI,185,"00000217, 1002415445, 1002415445С, 1003692596,..."
4,000003817,,Стартер Deutz Dingli BA28RT,154,"000003817, 3817"
5,000004697,,Датчик угла колес 000004697,223,"(LH)0110A10BN/HA(HC), 000004697, 00004697, 000..."
6,000004697,,Датчик угла колеса 000004697,223,"(LH)0110A10BN/HA(HC), 000004697, 00004697, 000..."
7,000004829,,Выхлопная труба 000004829,352,"000004829, 4829"
8,00000516A,,Фильтр гидравлический 00000516A,315,"00000516, 00000516A, 516, 516A, DH9426, R0060A..."
9,00000519,,Фильтр гидравлический 00000519,205,"00000519, 00000519A, 519, 519A"


,Артикул,Оригинальный номер,Номенклатура,Номер группы_old,Номер группы_new,Список аналогов_old,Список аналогов_new
0,2820302890,,Колесо 2820302890 для Подъемник ножничный Haul...,769,780,2820302890,2820302890
1,50009032,,"Колесо в сборе левое 50009032 (315/55D20 пена,...",1488,1493,"50009032, 50009032 Б/У","50009032, 50009032 Б/У"
2,50009032 Б/У,,Колесо в сборе левое б/у 50009032 (315/55D20 п...,1488,1493,"50009032, 50009032 Б/У","50009032, 50009032 Б/У"
3,50009033,,Колесо в сборе правое 50009033 (315/55D20 пена...,1489,1494,"50009033, 50009033 Б/У","50009033, 50009033 Б/У"
4,50009033 Б/У,,Колесо в сборе правое б/у 50009033 (315/55D20 ...,1489,1494,"50009033, 50009033 Б/У","50009033, 50009033 Б/У"
5,R1TG2-10821,,Гидромотор R1TG2-10821,956,964,R1TG2-10821,R1TG2-10821
